In [ ]:
# Run this cell first!
%pip install -q openai pydantic

from google.colab import userdata, drive
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
drive.mount("/content/drive")

# Build an Encyclopedia with AI

This notebook takes a list of texts and creates an encyclopedia entry for each:
- **Structured data** extracted by GPT-4o
- **Generated artwork** from DALL-E 3
- **JSON output** ready for your app

---

In [ ]:
import json
import requests
from pathlib import Path
from pydantic import BaseModel
from openai import OpenAI
from IPython.display import Image, display

client = OpenAI()

# Where to save outputs (customize this path)
OUTPUT_DIR = Path("/content/drive/MyDrive/encyclopedia")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ready! Files will be saved to: {OUTPUT_DIR}")

---

## 1. Your Texts

Add your passages here. Each needs an `id`, `name`, and `text`.

In [ ]:
texts = [
    {
        "id": "genesis",
        "name": "Genesis",
        "text": """In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. And God said, Let there be light: and there was light."""
    },
    {
        "id": "theogony",
        "name": "Hesiod's Theogony",
        "text": """First of all, the Void came into being, next broad-bosomed Earth, the solid and eternal home of all, and Eros, the most beautiful of the immortal gods, who in every man and every god softens the sinews and overpowers the prudent purpose of the mind."""
    },
    {
        "id": "metamorphoses",
        "name": "Ovid's Metamorphoses",
        "text": """Before the sea and lands began to be, before the sky had mantled everything, Nature displayed a single face, which they called Chaos: a raw and undivided mass, nothing but weight, lifeless, whose components were heaped together."""
    },
]

---

## 2. Define What You Want

This schema tells the AI what to extract. Customize the fields for your project.

In [ ]:
class Entry(BaseModel):
    title: str
    summary: str
    themes: list[str]
    era: str
    art_prompt: str  # Used to generate the image

---

## 3. The Two Functions

One extracts data, one generates art. That's it.

In [ ]:
def extract_entry(name: str, text: str) -> Entry:
    """Extract structured data from a text passage."""
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Create an encyclopedia entry for this passage from {name}.
For art_prompt, describe a vivid scene suitable for illustration.

{text}"""
        }],
        response_format=Entry
    )
    return response.choices[0].message.parsed


def generate_image(prompt: str, filename: str) -> str:
    """Generate an image with DALL-E and save it."""
    response = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size="1024x1024",
        quality="standard",
        n=1
    )
    
    # Download and save
    img_data = requests.get(response.data[0].url).content
    filepath = OUTPUT_DIR / filename
    filepath.write_bytes(img_data)
    return str(filepath)

---

## 4. Generate the Encyclopedia

Loop through all texts, extract data, generate images, show results.

In [ ]:
entries = []

for item in texts:
    print(f"Processing {item['name']}...")
    
    # Extract structured data
    entry = extract_entry(item["name"], item["text"])
    
    # Generate artwork
    image_file = f"{item['id']}.png"
    image_path = generate_image(entry.art_prompt, image_file)
    
    # Build the final entry
    entry_data = entry.model_dump()
    entry_data["id"] = item["id"]
    entry_data["source"] = item["name"]
    entry_data["image"] = image_file
    entry_data["original_text"] = item["text"]
    entries.append(entry_data)
    
    # Show result
    print(f"  -> {entry.title}")
    display(Image(filename=image_path, width=300))
    print()

print(f"Done! Created {len(entries)} entries.")

---

## 5. Save the Data

In [ ]:
output_file = OUTPUT_DIR / "entries.json"

with open(output_file, "w") as f:
    json.dump(entries, f, indent=2)

print(f"Saved to: {output_file}")
print()
print(json.dumps(entries[0], indent=2))

---

## Done!

Your files are in Google Drive:
- `encyclopedia/entries.json` — all the structured data
- `encyclopedia/*.png` — the generated images

**Next steps:**
1. Add more texts to the `texts` list
2. Customize the `Entry` schema for your needs
3. Use the JSON + images in your app